# OpenArXiv CS-100K FAISS RAG Experiment

This Colab notebook builds a persistent vector index for `open-index/open-arxiv` using `google/embeddinggemma-300m`, FAISS, and Google Drive.

The default experiment indexes `100,000` computer-science OpenArXiv papers. Each paper gets one vector from the full title + abstract, then the notebook runs retrieval and a Gemini-hosted RAG answer cell over the saved FAISS artifacts.

Persistent storage is written under:

`/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_<run_mode>/`

The FAISS index stores vectors. SQLite stores source metadata, abstract text, categories, and arXiv links needed for retrieval and RAG.


## Runtime notes

- Use a GPU runtime for embedding generation; an L4 is the intended baseline.
- Default `RUN_MODE = "cs_100k"` indexes up to `100,000` papers whose arXiv categories start with `cs.`.
- For a quick validation pass, set `RUN_MODE = "dry_run"`; it indexes up to `1,000` CS papers.
- For the full unfiltered corpus, set `RUN_MODE = "full"`, clear or move old artifacts, and expect many hours plus substantial Drive space.
- OpenArXiv includes many non-CS domains (`math`, `physics`, `q-bio`, `stat`, `econ`, and more). Filtering to CS makes the first real RAG test cheaper and more relevant to NLP/IR/ML queries.
- CS filtering is still a one-time linear dataset pass because OpenArXiv is a corpus table, not a category search index. The notebook saves the cvfgfgfiltered subset to Drive so later reconnects reload it directly.
- `google/embeddinggemma-300m` may require a Hugging Face token. Set `HF_TOKEN` before loading the model if access fails.
- Each indexed paper is embedded once as `title: <title> | text: <full abstract>`; abstracts are not chunked.
- The query side uses the EmbeddingGemma fact-check prompt: `task: fact checking | query: <question>`.
- `dry_run` defaults to `IndexIDMap2(IndexFlatIP)`: exact inner-product search over normalized embeddings, equivalent to cosine similarity.
- `cs_100k` defaults to `IndexIDMap2(IndexIVFFlat)`: approximate inverted-file search, useful for testing faster retrieval. Set `FAISS_INDEX_KIND = 'flat'` for an exact baseline.
- `full` defaults to `IndexIDMap2(IndexIVFPQ)` for compressed approximate search at larger scale.


## Repo dependency note

This notebook is intentionally standalone. You do **not** need to clone or install the `ScholarRAG` repo to build the FAISS index, run retrieval, or generate the optional Gemini RAG answer in Colab.

Some helper code is duplicated from the package on purpose: OpenArXiv normalization, text cleanup, EmbeddingGemma prompt formatting, arXiv URL creation, and FAISS/SQLite artifact handling. Keeping those helpers inline makes the long Colab indexing run reproducible even if the local app package changes.

Clone/install the repo only if you want to run the FastAPI app or import `scholarrag` modules inside the same Colab runtime after artifacts are built.


In [ ]:
# Colab dependency setup. Re-run this cell after switching runtimes.
!pip -q install -U datasets sentence-transformers transformers accelerate faiss-cpu pyarrow tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 53.7 MB/s eta 0:00:00


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import getpass
import os

if not os.environ.get('HF_TOKEN'):
    token = getpass.getpass('HF_TOKEN, optional unless the embedding model requires it: ').strip()
    if token:
        os.environ['HF_TOKEN'] = token

if not os.environ.get('GEMINI_API_KEY'):
    api_key = getpass.getpass('GEMINI_API_KEY for optional RAG generation, leave blank to skip: ').strip()
    if api_key:
        os.environ['GEMINI_API_KEY'] = api_key


Mounted at /content/drive
HF_TOKEN, optional unless the embedding model requires it: ··········
GEMINI_API_KEY for optional RAG generation, leave blank to skip: ··········


In [ ]:
from pathlib import Path
import os

DATASET_NAME = 'open-index/open-arxiv'
MODEL_ID = 'google/embeddinggemma-300m'
DIM = 768

DOCUMENT_MODE = 'title_abstract'
DOCUMENT_PROMPT_TEMPLATE = 'title: {title} | text: '
QUERY_PROMPT = 'task: fact checking | query: '

# Main experiment: index 100k CS papers. Use 'dry_run' first if you want a cheap validation pass.
RUN_MODE = 'full'  # 'dry_run', 'cs_100k', or 'full'
MAX_INDEXED_DOCUMENTS_BY_MODE = {
    'dry_run': 1_000,
    'cs_100k': 100_000,
    'full': None,
}
if RUN_MODE not in MAX_INDEXED_DOCUMENTS_BY_MODE:
    raise ValueError(f'Unsupported RUN_MODE: {RUN_MODE}')
MAX_INDEXED_DOCUMENTS = MAX_INDEXED_DOCUMENTS_BY_MODE[RUN_MODE]

# Category filter. Prefix 'cs.' includes cs.CL, cs.IR, cs.AI, cs.LG, etc.
# Leave both empty only for an unfiltered full-corpus run.
TARGET_CATEGORY_PREFIXES = ['cs.'] if RUN_MODE in {'dry_run', 'cs_100k'} else []
TARGET_CATEGORIES = []

# Recency filter. RAG-era queries need recent papers; otherwise the first 100k CS rows skew old.
MIN_UPDATE_DATE_BY_MODE = {
    'dry_run': '2000-01-01',
    'cs_100k': '2020-01-01',
    'full': '2020-01-01',
}
MIN_UPDATE_DATE = MIN_UPDATE_DATE_BY_MODE[RUN_MODE]
SORT_BY_RECENCY_BY_MODE = {
    'dry_run': False,
    'cs_100k': True,
    'full': True,
}
SORT_BY_RECENCY = SORT_BY_RECENCY_BY_MODE[RUN_MODE]

# Optional scan cap. None means scan until enough matching papers are indexed or the dataset ends.
MAX_SCAN_ROWS = None

# L4-oriented defaults for EmbeddingGemma over title+abstract texts.
# If Colab runs out of GPU memory, drop cs_100k/full to 256 or 128 and re-run from the last checkpoint.
BATCH_SIZE_BY_MODE = {
    'dry_run': 32,
    'cs_100k': 512,
    'full': 1024,
}
BATCH_SIZE = BATCH_SIZE_BY_MODE[RUN_MODE]

# CS filtering is a linear pass the first time, then reloads from Drive.
# Raise/lower FILTER_NUM_PROC if Colab CPU RAM becomes the bottleneck during filtering.
FILTER_BATCH_SIZE = 10_000
FILTER_NUM_PROC = min(8, max(1, os.cpu_count() or 1))
REBUILD_FILTERED_DATASET = False

# Search algorithm: 'flat' is exact; 'ivfflat' is fast approximate; 'ivfpq' is compressed approximate.
FAISS_INDEX_KIND_BY_MODE = {
    'dry_run': 'flat',
    'cs_100k': 'ivfflat',
    'full': 'ivfpq',
}
FAISS_INDEX_KIND = FAISS_INDEX_KIND_BY_MODE[RUN_MODE]
TRAINING_SAMPLE_RECORDS_BY_INDEX_KIND = {
    'flat': 0,
    'ivfflat': 20_000,
    'ivfpq': 50_000,
}
TRAINING_SAMPLE_RECORDS = TRAINING_SAMPLE_RECORDS_BY_INDEX_KIND[FAISS_INDEX_KIND]
NLIST_BY_MODE = {
    'dry_run': 64,
    'cs_100k': 512,
    'full': 4096,
}
NLIST = NLIST_BY_MODE[RUN_MODE]
PQ_M = 96
PQ_BITS = 8
NPROBE_BY_MODE = {
    'dry_run': 8,
    'cs_100k': 32,
    'full': 32,
}
NPROBE = NPROBE_BY_MODE[RUN_MODE]
SHOW_EMBEDDING_PROGRESS = True

# Durable checkpoints. Copying huge artifacts to Drive is slow, so keep only a few historical checkpoints.
CHECKPOINT_EVERY_RECORDS = 100_000
KEEP_CHECKPOINT_COPIES = 2
STOP_AFTER_CHECKPOINTS = None  # set to 1 for a manual checkpoint/resume test

DATE_SLICE = f"from_{MIN_UPDATE_DATE[:4]}_recent" if MIN_UPDATE_DATE else 'all_dates'
CORPUS_SLICE_NAME = f'{RUN_MODE}_{DATE_SLICE}'
DRIVE_ROOT = Path(f'/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_{CORPUS_SLICE_NAME}_{FAISS_INDEX_KIND}')
DRIVE_FAISS_DIR = DRIVE_ROOT / 'faiss'
DRIVE_METADATA_DIR = DRIVE_ROOT / 'metadata'
DRIVE_CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
DRIVE_DATASET_CACHE_DIR = DRIVE_ROOT / 'hf_cache'
DRIVE_FILTERED_DATASET_DIR = DRIVE_ROOT / 'filtered_dataset'
DRIVE_FILTERED_DATASET_MANIFEST_PATH = DRIVE_ROOT / 'filtered_dataset_manifest.json'
DRIVE_MANIFEST_PATH = DRIVE_ROOT / 'manifest.json'
DRIVE_FAISS_PATH = DRIVE_FAISS_DIR / 'open_arxiv_papers.faiss'
DRIVE_SQLITE_PATH = DRIVE_METADATA_DIR / 'open_arxiv_papers.sqlite'

LOCAL_ROOT = Path('/content/scholarrag_open_arxiv_work') / CORPUS_SLICE_NAME / FAISS_INDEX_KIND
LOCAL_FAISS_PATH = LOCAL_ROOT / 'open_arxiv_papers.faiss'
LOCAL_SQLITE_PATH = LOCAL_ROOT / 'open_arxiv_papers.sqlite'

for path in [DRIVE_FAISS_DIR, DRIVE_METADATA_DIR, DRIVE_CHECKPOINT_DIR, DRIVE_DATASET_CACHE_DIR, LOCAL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

DATASET_VIEW_ID = f"{DATASET_NAME}:{RUN_MODE}:scan={MAX_SCAN_ROWS}:categories={','.join(TARGET_CATEGORIES)}:prefixes={','.join(TARGET_CATEGORY_PREFIXES)}:min_update_date={MIN_UPDATE_DATE}:sort_recent={SORT_BY_RECENCY}"

print({
    'run_mode': RUN_MODE,
    'max_indexed_documents': MAX_INDEXED_DOCUMENTS,
    'batch_size': BATCH_SIZE,
    'faiss_index_kind': FAISS_INDEX_KIND,
    'nlist': NLIST,
    'nprobe': NPROBE,
    'training_sample_records': TRAINING_SAMPLE_RECORDS,
    'target_category_prefixes': TARGET_CATEGORY_PREFIXES,
    'target_categories': TARGET_CATEGORIES,
    'min_update_date': MIN_UPDATE_DATE,
    'sort_by_recency': SORT_BY_RECENCY,
    'filter_num_proc': FILTER_NUM_PROC,
    'drive_root': str(DRIVE_ROOT),
})


{'run_mode': 'full', 'max_indexed_documents': None, 'batch_size': 1024, 'faiss_index_kind': 'ivfpq', 'nlist': 4096, 'nprobe': 32, 'training_sample_records': 50000, 'target_category_prefixes': [], 'target_categories': [], 'min_update_date': '2020-01-01', 'sort_by_recency': True, 'filter_num_proc': 8, 'drive_root': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq'}


In [ ]:
import json
import math
import shutil
import sqlite3
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from typing import Any, Iterable
from uuid import NAMESPACE_URL, uuid5

import faiss
import numpy as np
from datasets import load_dataset, load_from_disk
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def load_manifest() -> dict[str, Any]:
    if DRIVE_MANIFEST_PATH.exists():
        return json.loads(DRIVE_MANIFEST_PATH.read_text())
    return {}

def write_manifest(manifest: dict[str, Any]) -> None:
    manifest = dict(manifest)
    manifest['updated_at'] = utc_now()
    tmp_path = DRIVE_MANIFEST_PATH.with_suffix('.json.tmp')
    tmp_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
    tmp_path.replace(DRIVE_MANIFEST_PATH)

manifest = load_manifest()
manifest

{'checkpoint_number': 14,
 'dataset_name': 'open-index/open-arxiv',
 'dataset_view_id': 'open-index/open-arxiv:full:scan=None:categories=:prefixes=:min_update_date=2020-01-01:sort_recent=True',
 'document_mode': 'title_abstract',
 'document_prompt_template': 'title: {title} | text: ',
 'embedding_dimension': 768,
 'faiss_index_kind': 'ivfpq',
 'faiss_index_type': 'IndexIDMap2(Index)',
 'faiss_nlist_actual': 0,
 'faiss_nlist_requested': 4096,
 'faiss_path': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/faiss/open_arxiv_papers.faiss',
 'faiss_pq_bits': 8,
 'faiss_pq_m': 96,
 'index_dataset_rows': 1454945,
 'indexed_documents': 1354576,
 'indexed_records': 1354576,
 'last_checkpoint_faiss': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/checkpoints/00014_000001354576_papers.faiss',
 'last_checkpoint_sqlite': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full

In [ ]:
@dataclass(frozen=True)
class Paper:
    paper_id: str
    title: str
    abstract: str
    categories: list[str]
    update_date: str | None
    authors: list[str]

def clean_text(value: str | None) -> str:
    return ' '.join((value or '').split())

def arxiv_abs_url(paper_id: str) -> str:
    return f'https://arxiv.org/abs/{paper_id}'

def parse_authors(authors_parsed: str | None) -> list[str]:
    if not authors_parsed:
        return []
    try:
        raw_authors = json.loads(authors_parsed)
    except json.JSONDecodeError:
        return []
    authors = []
    for author in raw_authors:
        if not isinstance(author, list) or len(author) < 2:
            continue
        last = str(author[0] or '').strip()
        first = str(author[1] or '').strip()
        name = ' '.join(part for part in [first, last] if part)
        if name:
            authors.append(name)
    return authors

def paper_matches_target_categories(paper: Paper) -> bool:
    if not TARGET_CATEGORIES and not TARGET_CATEGORY_PREFIXES:
        return True
    categories = set(paper.categories)
    if TARGET_CATEGORIES and categories.intersection(TARGET_CATEGORIES):
        return True
    return any(
        category.startswith(prefix)
        for category in paper.categories
        for prefix in TARGET_CATEGORY_PREFIXES
    )

def categories_match_target(categories_text: str | None) -> bool:
    if not TARGET_CATEGORIES and not TARGET_CATEGORY_PREFIXES:
        return True
    categories = [c.strip() for c in str(categories_text or '').split() if c.strip()]
    if TARGET_CATEGORIES and set(categories).intersection(TARGET_CATEGORIES):
        return True
    return any(
        category.startswith(prefix)
        for category in categories
        for prefix in TARGET_CATEGORY_PREFIXES
    )

def date_matches_min_update_date(update_date: str | None) -> bool:
    if not MIN_UPDATE_DATE:
        return True
    value = str(update_date or '').strip()
    return bool(value) and value >= MIN_UPDATE_DATE

def record_matches_filters(categories_text: str | None, update_date: str | None) -> bool:
    return categories_match_target(categories_text) and date_matches_min_update_date(update_date)

def records_match_filters(batch: dict[str, list[Any]]) -> list[bool]:
    categories = batch.get('categories', [])
    update_dates = batch.get('update_date', [])
    return [record_matches_filters(category_text, update_date) for category_text, update_date in zip(categories, update_dates)]

def normalize_record(record: dict[str, Any]) -> Paper | None:
    paper_id = str(record.get('id') or '').strip()
    title = clean_text(record.get('title'))
    abstract = clean_text(record.get('abstract'))
    if not paper_id or not abstract:
        return None
    categories = [c.strip() for c in str(record.get('categories') or '').split() if c.strip()]
    update_date = str(record.get('update_date') or '').strip() or None
    authors = parse_authors(record.get('authors_parsed'))
    paper = Paper(paper_id=paper_id, title=title, abstract=abstract, categories=categories, update_date=update_date, authors=authors)
    if not paper_matches_target_categories(paper):
        return None
    if not date_matches_min_update_date(update_date):
        return None
    return paper

def stable_point_id(paper_id: str) -> str:
    return str(uuid5(NAMESPACE_URL, f'scholarrag:open-arxiv:{paper_id}:abstract'))

def paper_text(paper: Paper) -> str:
    return paper.abstract

def format_document_for_embedding(paper: Paper) -> str:
    title = clean_text(paper.title) or 'none'
    return DOCUMENT_PROMPT_TEMPLATE.format(title=title) + paper_text(paper).strip()

def format_query_for_embedding(query: str) -> str:
    return QUERY_PROMPT + query.strip()


In [ ]:
def connect_sqlite(path: Path = LOCAL_SQLITE_PATH) -> sqlite3.Connection:
    conn = sqlite3.connect(path)
    conn.row_factory = sqlite3.Row
    conn.execute('pragma journal_mode=delete')
    conn.execute('pragma synchronous=normal')
    conn.execute('pragma temp_store=memory')
    return conn

def init_sqlite(conn: sqlite3.Connection) -> None:
    conn.executescript(
        """
        create table if not exists papers (
            vector_id integer primary key,
            point_id text unique not null,
            paper_id text not null,
            title text not null,
            text text not null,
            categories_json text not null,
            update_date text,
            authors_json text not null,
            created_at text not null
        );
        create index if not exists idx_papers_paper_id on papers(paper_id);
        create index if not exists idx_papers_update_date on papers(update_date);
        create table if not exists indexing_state (
            key text primary key,
            value text not null
        );
        """
    )
    conn.commit()

def insert_metadata(conn: sqlite3.Connection, vector_ids: np.ndarray, papers: list[Paper]) -> None:
    rows = [
        (
            int(vector_id),
            stable_point_id(paper.paper_id),
            paper.paper_id,
            paper.title,
            paper_text(paper),
            json.dumps(paper.categories),
            paper.update_date,
            json.dumps(paper.authors),
            utc_now(),
        )
        for vector_id, paper in zip(vector_ids, papers)
    ]
    with conn:
        conn.executemany(
            """
            insert or replace into papers (
                vector_id, point_id, paper_id, title, text,
                categories_json, update_date, authors_json, created_at
            ) values (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            rows,
        )

def fetch_papers_by_vector_ids(conn: sqlite3.Connection, vector_ids: Iterable[int]) -> list[dict[str, Any]]:
    ids = [int(x) for x in vector_ids if int(x) >= 0]
    if not ids:
        return []
    placeholders = ','.join('?' for _ in ids)
    rows = conn.execute(f'select * from papers where vector_id in ({placeholders})', ids).fetchall()
    by_id = {int(row['vector_id']): dict(row) for row in rows}
    ordered = []
    for vector_id in ids:
        row = by_id.get(vector_id)
        if row:
            row['categories'] = json.loads(row.pop('categories_json') or '[]')
            row['authors'] = json.loads(row.pop('authors_json') or '[]')
            row['chunk_id'] = 0
            row['arxiv_url'] = arxiv_abs_url(row['paper_id'])
            ordered.append(row)
    return ordered


In [ ]:
print('Loading dataset. First full download/cache can take a while...')
dataset = load_dataset(DATASET_NAME, split='train', cache_dir=str(DRIVE_DATASET_CACHE_DIR))
source_row_limit = len(dataset) if MAX_SCAN_ROWS is None else min(MAX_SCAN_ROWS, len(dataset))
source_dataset = dataset if source_row_limit == len(dataset) else dataset.select(range(source_row_limit))

needs_dataset_filter = bool(TARGET_CATEGORIES or TARGET_CATEGORY_PREFIXES or MIN_UPDATE_DATE)
if needs_dataset_filter:
    filtered_cache_manifest = {}
    if DRIVE_FILTERED_DATASET_MANIFEST_PATH.exists():
        filtered_cache_manifest = json.loads(DRIVE_FILTERED_DATASET_MANIFEST_PATH.read_text())
    filtered_cache_matches = (
        DRIVE_FILTERED_DATASET_DIR.exists()
        and filtered_cache_manifest.get('dataset_view_id') == DATASET_VIEW_ID
        and not REBUILD_FILTERED_DATASET
    )
    if filtered_cache_matches:
        print(f'Loading cached filtered dataset from {DRIVE_FILTERED_DATASET_DIR}')
        filtered_dataset = load_from_disk(str(DRIVE_FILTERED_DATASET_DIR))
    else:
        if DRIVE_FILTERED_DATASET_DIR.exists():
            shutil.rmtree(DRIVE_FILTERED_DATASET_DIR)
        print('Filtering dataset by arXiv categories/date. This is a one-time linear pass; the result is saved to Drive.')
        filtered_dataset = source_dataset.filter(
            records_match_filters,
            batched=True,
            batch_size=FILTER_BATCH_SIZE,
            num_proc=FILTER_NUM_PROC,
            desc='Filtering target categories/date',
        )
        if SORT_BY_RECENCY:
            print('Sorting filtered dataset by update_date descending so the 100k slice is recent...')
            filtered_dataset = filtered_dataset.sort('update_date', reverse=True)
        filtered_dataset.save_to_disk(str(DRIVE_FILTERED_DATASET_DIR))
        DRIVE_FILTERED_DATASET_MANIFEST_PATH.write_text(json.dumps({
            'dataset_view_id': DATASET_VIEW_ID,
            'dataset_name': DATASET_NAME,
            'source_dataset_rows': source_row_limit,
            'filtered_dataset_rows': len(filtered_dataset),
            'target_categories': TARGET_CATEGORIES,
            'target_category_prefixes': TARGET_CATEGORY_PREFIXES,
            'min_update_date': MIN_UPDATE_DATE,
            'sort_by_recency': SORT_BY_RECENCY,
            'created_at': utc_now(),
        }, indent=2, sort_keys=True))
        print(f'Saved filtered dataset to {DRIVE_FILTERED_DATASET_DIR}')
else:
    filtered_dataset = source_dataset

index_dataset_size = len(filtered_dataset)
row_limit = index_dataset_size if MAX_INDEXED_DOCUMENTS is None else min(MAX_INDEXED_DOCUMENTS, index_dataset_size)
index_dataset = filtered_dataset if row_limit == index_dataset_size else filtered_dataset.select(range(row_limit))
print(
    f'Dataset rows available: {len(dataset):,}; source scan cap: {source_row_limit:,}; '
    f'filtered/indexable rows: {index_dataset_size:,}; rows selected for this run: {len(index_dataset):,}'
)
if len(index_dataset):
    newest = index_dataset[0].get('update_date')
    oldest_selected = index_dataset[len(index_dataset) - 1].get('update_date')
    print(f'Selected date range: newest={newest}, oldest_selected={oldest_selected}')


print('Loading embedding model...')
model_kwargs = {}
try:
    import torch
    if torch.cuda.is_available():
        model_kwargs['torch_dtype'] = torch.bfloat16
except Exception:
    model_kwargs = {}

embedder = SentenceTransformer(
    MODEL_ID,
    token=os.environ.get('HF_TOKEN') or None,
    truncate_dim=DIM,
    model_kwargs=model_kwargs,
)

def encode_papers(papers: list[Paper], *, show_progress: bool | None = None) -> np.ndarray:
    texts = [format_document_for_embedding(paper) for paper in papers]
    vectors = embedder.encode(
        texts,
        batch_size=BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=SHOW_EMBEDDING_PROGRESS if show_progress is None else show_progress,
    )
    vectors = np.asarray(vectors, dtype=np.float32)
    if vectors.ndim != 2 or vectors.shape[1] != DIM:
        raise ValueError(f'Embedding shape mismatch: expected (*, {DIM}), got {vectors.shape}')
    return vectors

def encode_query(query: str) -> np.ndarray:
    vector = embedder.encode(
        [format_query_for_embedding(query)],
        batch_size=1,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return np.asarray(vector, dtype=np.float32)

Loading dataset. First full download/cache can take a while...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/417 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/417 [00:00<?, ?it/s]

Loading cached filtered dataset from /content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/filtered_dataset
Dataset rows available: 2,989,022; source scan cap: 2,989,022; filtered/indexable rows: 1,454,945; rows selected for this run: 1,454,945
Selected date range: newest=2026-03-20, oldest_selected=2020-01-01
Loading embedding model...


...

config.json:   0%|          | 0.00/312 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

In [ ]:
def choose_nlist(train_count: int) -> int:
    if train_count < 1_000:
        return max(8, min(64, train_count // 8 or 8))
    enough_training = max(16, train_count // 39)
    return int(min(NLIST, enough_training))

def describe_faiss_index(index: faiss.Index) -> str:
    base = index.index if isinstance(index, faiss.IndexIDMap2) else index
    if isinstance(base, faiss.IndexFlatIP):
        return 'IndexIDMap2(IndexFlatIP) exact cosine-via-normalized-inner-product'
    if isinstance(base, faiss.IndexIVFFlat):
        return 'IndexIDMap2(IndexIVFFlat) approximate cosine-via-normalized-inner-product'
    if isinstance(base, faiss.IndexIVFPQ):
        return 'IndexIDMap2(IndexIVFPQ) compressed approximate cosine-via-normalized-inner-product'
    return f'IndexIDMap2({type(base).__name__})'

def build_faiss_index(train_vectors: np.ndarray | None = None) -> faiss.IndexIDMap2:
    if FAISS_INDEX_KIND == 'flat':
        print(f'Creating exact IndexFlatIP for {DIM}-d normalized vectors')
        return faiss.IndexIDMap2(faiss.IndexFlatIP(DIM))
    if train_vectors is None or len(train_vectors) == 0:
        raise ValueError(f'{FAISS_INDEX_KIND} requires non-empty training vectors')
    if train_vectors.dtype != np.float32:
        train_vectors = train_vectors.astype(np.float32)
    effective_nlist = choose_nlist(len(train_vectors))
    quantizer = faiss.IndexFlatIP(DIM)
    if FAISS_INDEX_KIND == 'ivfflat':
        base_index = faiss.IndexIVFFlat(quantizer, DIM, effective_nlist, faiss.METRIC_INNER_PRODUCT)
        print(f'Training IndexIVFFlat with nlist={effective_nlist}, train_vectors={len(train_vectors):,}')
    elif FAISS_INDEX_KIND == 'ivfpq':
        base_index = faiss.IndexIVFPQ(quantizer, DIM, effective_nlist, PQ_M, PQ_BITS, faiss.METRIC_INNER_PRODUCT)
        print(f'Training IndexIVFPQ with nlist={effective_nlist}, m={PQ_M}, nbits={PQ_BITS}, train_vectors={len(train_vectors):,}')
    else:
        raise ValueError(f'Unsupported FAISS_INDEX_KIND: {FAISS_INDEX_KIND}')
    train_start = time.perf_counter()
    base_index.train(train_vectors)
    print(f'FAISS training finished in {time.perf_counter() - train_start:.1f}s')
    base_index.nprobe = min(NPROBE, effective_nlist)
    return faiss.IndexIDMap2(base_index)

def set_nprobe(index: faiss.Index, nprobe: int = NPROBE) -> None:
    base = index.index if isinstance(index, faiss.IndexIDMap2) else index
    if hasattr(base, 'nprobe'):
        base.nprobe = min(int(nprobe), int(getattr(base, 'nlist', nprobe)))

def load_faiss_index(path: Path = LOCAL_FAISS_PATH) -> faiss.IndexIDMap2:
    index = faiss.read_index(str(path))
    set_nprobe(index, NPROBE)
    return index

def add_vectors(index: faiss.IndexIDMap2, conn: sqlite3.Connection, papers: list[Paper], next_vector_id: int) -> int:
    if not papers:
        return next_vector_id
    vectors = encode_papers(papers, show_progress=False)
    vector_ids = np.arange(next_vector_id, next_vector_id + len(papers), dtype=np.int64)
    index.add_with_ids(vectors, vector_ids)
    insert_metadata(conn, vector_ids, papers)
    return next_vector_id + len(papers)

In [ ]:
def restore_artifacts_from_drive() -> None:
    if DRIVE_FAISS_PATH.exists():
        shutil.copy2(DRIVE_FAISS_PATH, LOCAL_FAISS_PATH)
        print(f'Restored FAISS index from {DRIVE_FAISS_PATH}')
    if DRIVE_SQLITE_PATH.exists():
        shutil.copy2(DRIVE_SQLITE_PATH, LOCAL_SQLITE_PATH)
        print(f'Restored SQLite metadata from {DRIVE_SQLITE_PATH}')

def prune_old_checkpoints(keep: int = KEEP_CHECKPOINT_COPIES) -> None:
    if keep is None or keep <= 0:
        return
    faiss_checkpoints = sorted(DRIVE_CHECKPOINT_DIR.glob('*.faiss'))
    sqlite_checkpoints = sorted(DRIVE_CHECKPOINT_DIR.glob('*.sqlite'))
    for old_path in faiss_checkpoints[:-keep] + sqlite_checkpoints[:-keep]:
        old_path.unlink(missing_ok=True)

def checkpoint_artifacts(
    *,
    index: faiss.IndexIDMap2,
    conn: sqlite3.Connection,
    next_row_index: int,
    next_vector_id: int,
    indexed_records: int,
    indexed_documents: int,
    checkpoint_number: int,
    status: str,
) -> None:
    checkpoint_start = time.perf_counter()
    print(f'Saving checkpoint {checkpoint_number} to local disk and Drive...')
    faiss.write_index(index, str(LOCAL_FAISS_PATH))
    sqlite_backup_path = LOCAL_ROOT / 'open_arxiv_papers.backup.sqlite'
    if sqlite_backup_path.exists():
        sqlite_backup_path.unlink()
    backup_conn = sqlite3.connect(sqlite_backup_path)
    conn.backup(backup_conn)
    backup_conn.close()

    shutil.copy2(LOCAL_FAISS_PATH, DRIVE_FAISS_PATH)
    shutil.copy2(sqlite_backup_path, DRIVE_SQLITE_PATH)

    checkpoint_tag = f'{checkpoint_number:05d}_{indexed_documents:012d}_papers'
    checkpoint_faiss = DRIVE_CHECKPOINT_DIR / f'{checkpoint_tag}.faiss'
    checkpoint_sqlite = DRIVE_CHECKPOINT_DIR / f'{checkpoint_tag}.sqlite'
    shutil.copy2(LOCAL_FAISS_PATH, checkpoint_faiss)
    shutil.copy2(sqlite_backup_path, checkpoint_sqlite)

    prune_old_checkpoints()

    write_manifest(
        {
            'status': status,
            'dataset_name': DATASET_NAME,
            'dataset_view_id': DATASET_VIEW_ID,
            'model_id': MODEL_ID,
            'embedding_dimension': DIM,
            'document_prompt_template': DOCUMENT_PROMPT_TEMPLATE,
            'query_prompt': QUERY_PROMPT,
            'document_mode': DOCUMENT_MODE,
            'faiss_index_kind': FAISS_INDEX_KIND,
            'faiss_index_type': describe_faiss_index(index),
            'faiss_nlist_requested': NLIST,
            'faiss_nlist_actual': int(getattr(index.index if isinstance(index, faiss.IndexIDMap2) else index, 'nlist', 0)),
            'faiss_pq_m': PQ_M,
            'faiss_pq_bits': PQ_BITS,
            'nprobe': NPROBE,
            'run_mode': RUN_MODE,
            'max_scan_rows': MAX_SCAN_ROWS,
            'source_dataset_rows': source_row_limit,
            'index_dataset_rows': len(index_dataset),
            'max_indexed_documents': MAX_INDEXED_DOCUMENTS,
            'target_categories': TARGET_CATEGORIES,
            'target_category_prefixes': TARGET_CATEGORY_PREFIXES,
            'min_update_date': MIN_UPDATE_DATE,
            'sort_by_recency': SORT_BY_RECENCY,
            'next_row_index': next_row_index,
            'next_vector_id': next_vector_id,
            'indexed_documents': indexed_documents,
            'indexed_records': indexed_records,
            'checkpoint_number': checkpoint_number,
            'faiss_path': str(DRIVE_FAISS_PATH),
            'sqlite_path': str(DRIVE_SQLITE_PATH),
            'last_checkpoint_faiss': str(checkpoint_faiss),
            'last_checkpoint_sqlite': str(checkpoint_sqlite),
        }
    )
    print(f'Checkpoint {checkpoint_number} saved in {time.perf_counter() - checkpoint_start:.1f}s: documents={indexed_documents:,}, records={indexed_records:,}, next_row={next_row_index:,}')


In [ ]:
def paper_for_row(row_index: int) -> Paper | None:
    return normalize_record(index_dataset[int(row_index)])

def target_document_count_reached(indexed_documents: int) -> bool:
    return MAX_INDEXED_DOCUMENTS is not None and indexed_documents >= MAX_INDEXED_DOCUMENTS

def collect_training_papers(start_row_index: int, row_limit: int) -> tuple[list[Paper], int]:
    target = min(TRAINING_SAMPLE_RECORDS, row_limit - start_row_index)
    if target <= 0:
        return [], start_row_index
    papers: list[Paper] = []
    next_row_index = start_row_index
    progress = tqdm(total=target, desc=f'Collecting {FAISS_INDEX_KIND} training papers')
    try:
        for row_index in range(start_row_index, row_limit):
            paper = paper_for_row(row_index)
            if paper is not None:
                papers.append(paper)
                progress.update(1)
            next_row_index = row_index + 1
            progress.set_postfix(scanned=next_row_index, matching_papers=len(papers))
            if len(papers) >= target:
                break
    finally:
        progress.close()
    if not papers:
        raise RuntimeError('No papers were collected for FAISS training. Check category filters and scan cap.')
    print(f'Collected {len(papers):,} training papers after scanning {next_row_index - start_row_index:,} rows')
    return papers, next_row_index

def run_indexing() -> dict[str, Any]:
    restore_artifacts_from_drive()
    current_manifest = load_manifest()
    start_row_index = int(current_manifest.get('next_row_index', 0))
    next_vector_id = int(current_manifest.get('next_vector_id', 0))
    indexed_documents = int(current_manifest.get('indexed_documents', 0))
    indexed_records = int(current_manifest.get('indexed_records', 0))
    checkpoint_number = int(current_manifest.get('checkpoint_number', 0))

    if current_manifest and current_manifest.get('run_mode') != RUN_MODE:
        raise RuntimeError(
            f"Existing artifacts were created with run_mode={current_manifest.get('run_mode')!r}, "
            f"but the current config has RUN_MODE={RUN_MODE!r}. Move or delete the Drive artifact folder "
            'before starting a different run mode.'
        )

    if current_manifest and int(current_manifest.get('indexed_records', 0)) > 0 and current_manifest.get('dataset_view_id') != DATASET_VIEW_ID:
        raise RuntimeError(
            'Existing artifacts were created with an older or different dataset view. Move or delete the Drive artifact folder '
            'before resuming with the cached filtered dataset view.'
        )

    if current_manifest and int(current_manifest.get('indexed_records', 0)) > 0 and current_manifest.get('faiss_index_kind') != FAISS_INDEX_KIND:
        raise RuntimeError(
            f"Existing artifacts use faiss_index_kind={current_manifest.get('faiss_index_kind')!r}, "
            f"but the current config uses {FAISS_INDEX_KIND!r}. Use the matching Drive folder or rebuild artifacts."
        )

    conn = connect_sqlite()
    init_sqlite(conn)

    if LOCAL_FAISS_PATH.exists() and current_manifest:
        index = load_faiss_index(LOCAL_FAISS_PATH)
        print(f'Resuming from row={start_row_index:,}, next_vector_id={next_vector_id:,}, existing_vectors={index.ntotal:,}')
    else:
        start_row_index = 0
        next_vector_id = 0
        indexed_documents = 0
        indexed_records = 0
        checkpoint_number = 0
        if FAISS_INDEX_KIND == 'flat':
            index = build_faiss_index()
            print('Starting exact flat index immediately; no training sample is needed.')
        else:
            training_papers, start_row_index = collect_training_papers(0, row_limit)
            print(f'Embedding {len(training_papers):,} training papers with batch_size={BATCH_SIZE}...')
            embed_start = time.perf_counter()
            training_vectors = encode_papers(training_papers, show_progress=True)
            print(f'Training embeddings finished in {time.perf_counter() - embed_start:.1f}s')
            index = build_faiss_index(training_vectors)
            print(f'Adding {len(training_papers):,} training vectors to FAISS and SQLite...')
            add_start = time.perf_counter()
            vector_ids = np.arange(0, len(training_papers), dtype=np.int64)
            index.add_with_ids(training_vectors, vector_ids)
            insert_metadata(conn, vector_ids, training_papers)
            print(f'Added training vectors in {time.perf_counter() - add_start:.1f}s')
            next_vector_id = len(training_papers)
            indexed_records = len(training_papers)
            indexed_documents = len(training_papers)
            checkpoint_number += 1
            checkpoint_artifacts(
                index=index,
                conn=conn,
                next_row_index=start_row_index,
                next_vector_id=next_vector_id,
                indexed_records=indexed_records,
                indexed_documents=indexed_documents,
                checkpoint_number=checkpoint_number,
                status='running',
            )
            if STOP_AFTER_CHECKPOINTS == 1:
                conn.close()
                return load_manifest()

    pending_papers: list[Paper] = []
    records_at_last_checkpoint = indexed_records
    checkpoints_this_call = 0
    start_time = time.time()

    progress = tqdm(range(start_row_index, row_limit), initial=start_row_index, total=row_limit, desc='Indexing OpenArXiv')
    for row_index in progress:
        if target_document_count_reached(indexed_documents):
            start_row_index = row_index
            break

        paper = paper_for_row(row_index)
        if paper is not None:
            pending_papers.append(paper)
            indexed_documents += 1

        while len(pending_papers) >= BATCH_SIZE:
            batch = pending_papers[:BATCH_SIZE]
            pending_papers = pending_papers[BATCH_SIZE:]
            next_vector_id = add_vectors(index, conn, batch, next_vector_id)
            indexed_records += len(batch)

        progress.set_postfix(documents=indexed_documents, records=indexed_records)

        if indexed_records - records_at_last_checkpoint >= CHECKPOINT_EVERY_RECORDS:
            checkpoint_number += 1
            checkpoint_artifacts(
                index=index,
                conn=conn,
                next_row_index=row_index + 1,
                next_vector_id=next_vector_id,
                indexed_records=indexed_records,
                indexed_documents=indexed_documents,
                checkpoint_number=checkpoint_number,
                status='running',
            )
            checkpoints_this_call += 1
            records_at_last_checkpoint = indexed_records
            if STOP_AFTER_CHECKPOINTS is not None and checkpoints_this_call >= STOP_AFTER_CHECKPOINTS:
                conn.close()
                return load_manifest()
    else:
        start_row_index = row_limit

    while pending_papers:
        batch = pending_papers[:BATCH_SIZE]
        pending_papers = pending_papers[BATCH_SIZE:]
        next_vector_id = add_vectors(index, conn, batch, next_vector_id)
        indexed_records += len(batch)

    status = 'complete' if start_row_index >= len(index_dataset) or target_document_count_reached(indexed_documents) else 'partial_complete'
    checkpoint_number += 1
    checkpoint_artifacts(
        index=index,
        conn=conn,
        next_row_index=start_row_index,
        next_vector_id=next_vector_id,
        indexed_records=indexed_records,
        indexed_documents=indexed_documents,
        checkpoint_number=checkpoint_number,
        status=status,
    )
    conn.close()
    elapsed = time.time() - start_time
    print(f'Indexing call complete in {elapsed / 60:.1f} minutes')
    return load_manifest()


In [ ]:
# Run this cell to build or resume the paper-level FAISS index.
# Dry run defaults to 1,000 rows. For the full run, set RUN_MODE = 'full' in the config cell.
result_manifest = run_indexing()
result_manifest


Restored FAISS index from /content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/faiss/open_arxiv_papers.faiss
Restored SQLite metadata from /content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/metadata/open_arxiv_papers.sqlite
Resuming from row=1,354,576, next_vector_id=1,354,576, existing_vectors=1,354,576


Indexing OpenArXiv:  93%|#########3| 1354576/1454945 [00:00<?, ?it/s]

Saving checkpoint 15 to local disk and Drive...
Checkpoint 15 saved in 34.7s: documents=1,454,928, records=1,454,928, next_row=1,454,928
Saving checkpoint 16 to local disk and Drive...
Checkpoint 16 saved in 67.6s: documents=1,454,945, records=1,454,945, next_row=1,454,945
Indexing call complete in 8.9 minutes


{'checkpoint_number': 16,
 'dataset_name': 'open-index/open-arxiv',
 'dataset_view_id': 'open-index/open-arxiv:full:scan=None:categories=:prefixes=:min_update_date=2020-01-01:sort_recent=True',
 'document_mode': 'title_abstract',
 'document_prompt_template': 'title: {title} | text: ',
 'embedding_dimension': 768,
 'faiss_index_kind': 'ivfpq',
 'faiss_index_type': 'IndexIDMap2(Index)',
 'faiss_nlist_actual': 0,
 'faiss_nlist_requested': 4096,
 'faiss_path': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/faiss/open_arxiv_papers.faiss',
 'faiss_pq_bits': 8,
 'faiss_pq_m': 96,
 'index_dataset_rows': 1454945,
 'indexed_documents': 1454945,
 'indexed_records': 1454945,
 'last_checkpoint_faiss': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full_from_2020_recent_ivfpq/checkpoints/00016_000001454945_papers.faiss',
 'last_checkpoint_sqlite': '/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_full

## Run modes

The notebook defaults to `RUN_MODE = "cs_100k"`, which indexes up to 100,000 OpenArXiv papers with at least one category beginning with `cs.`.

For a cheap smoke test, set `RUN_MODE = "dry_run"` and rerun from the config cell onward. For an unfiltered corpus build, set `RUN_MODE = "full"`, move or delete old artifacts for that run mode, and expect a much longer job.

If you want to test checkpoint/resume before a long run, set `STOP_AFTER_CHECKPOINTS = 1`, run the indexing cell, restart the runtime, then set `STOP_AFTER_CHECKPOINTS = None` and rerun.


In [ ]:
def validate_manifest() -> None:
    m = load_manifest()
    required = {
        'dataset_name': DATASET_NAME,
        'model_id': MODEL_ID,
        'embedding_dimension': DIM,
        'document_prompt_template': DOCUMENT_PROMPT_TEMPLATE,
        'query_prompt': QUERY_PROMPT,
        'target_category_prefixes': TARGET_CATEGORY_PREFIXES,
        'target_categories': TARGET_CATEGORIES,
        'document_mode': DOCUMENT_MODE,
        'min_update_date': MIN_UPDATE_DATE,
        'sort_by_recency': SORT_BY_RECENCY,
    }
    for key, expected in required.items():
        actual = m.get(key)
        assert actual == expected, f'{key}: expected {expected!r}, got {actual!r}'
    assert m.get('faiss_index_kind') == FAISS_INDEX_KIND, m.get('faiss_index_kind')
    assert int(m.get('indexed_records', 0)) > 0
    print('Manifest validation passed')

def validate_artifacts() -> None:
    assert DRIVE_FAISS_PATH.exists(), DRIVE_FAISS_PATH
    assert DRIVE_SQLITE_PATH.exists(), DRIVE_SQLITE_PATH
    index = faiss.read_index(str(DRIVE_FAISS_PATH))
    conn = sqlite3.connect(DRIVE_SQLITE_PATH)
    row = conn.execute('select count(*) from papers').fetchone()[0]
    conn.close()
    assert index.ntotal == row, f'FAISS vectors {index.ntotal:,} != SQLite rows {row:,}'
    print(f'Artifact validation passed: {index.ntotal:,} paper vectors and metadata rows')

validate_manifest()
validate_artifacts()


Manifest validation passed


AssertionError: FAISS vectors 1,454,945 != SQLite rows 1,454,934

In [ ]:
# Load durable artifacts for retrieval. This cell can run in a fresh Colab session after mounting Drive.
retrieval_index = faiss.read_index(str(DRIVE_FAISS_PATH))
set_nprobe(retrieval_index, NPROBE)
retrieval_conn = connect_sqlite(DRIVE_SQLITE_PATH)
print(f'Loaded retrieval index with {retrieval_index.ntotal:,} vectors')

Loaded retrieval index with 1,454,945 vectors


In [ ]:
def search_open_arxiv(question: str, *, top_k: int = 10, nprobe: int = NPROBE) -> list[dict[str, Any]]:
    set_nprobe(retrieval_index, nprobe)
    query_vector = encode_query(question)
    search_start = time.perf_counter()
    scores, ids = retrieval_index.search(query_vector, top_k)
    search_ms = (time.perf_counter() - search_start) * 1000
    vector_ids = [int(x) for x in ids[0].tolist() if int(x) >= 0]
    rows = fetch_papers_by_vector_ids(retrieval_conn, vector_ids)
    row_by_id = {int(row['vector_id']): row for row in rows}
    results = []
    for rank, (score, vector_id) in enumerate(zip(scores[0].tolist(), ids[0].tolist()), start=1):
        vector_id = int(vector_id)
        if vector_id < 0:
            continue
        row = row_by_id.get(vector_id)
        if row is None:
            raise RuntimeError(f'Missing SQLite metadata for vector_id={vector_id}')
        results.append({**row, 'rank': rank, 'score': float(score), 'search_ms': search_ms, 'arxiv_url': arxiv_abs_url(row['paper_id'])})
    return results

def print_results(results: list[dict[str, Any]]) -> None:
    for result in results:
        categories = ' '.join(result.get('categories') or [])
        preview = clean_text(result['text'])[:500]
        latency = result.get('search_ms')
        latency_text = f' search_ms={latency:.1f}' if latency is not None and result['rank'] == 1 else ''
        print(f"[{result['rank']}] score={result['score']:.4f}{latency_text} paper_id={result['paper_id']} paper-level")
        print(f"title: {result['title']}")
        print(f"url: {result['arxiv_url']}")
        print(f"categories: {categories} updated={result.get('update_date')}")
        print(f"preview: {preview}\n")

question = 'tensor parallelism for multimodal models?'
results = search_open_arxiv(question, top_k=10)
print_results(results)


[1] score=0.6157 search_ms=4.8 paper_id=2309.14118 paper-level
title: MultiModN- Multimodal, Multi-Task, Interpretable Modular Networks
url: https://arxiv.org/abs/2309.14118
categories: cs.LG updated=2023-11-07
preview: Predicting multiple real-world tasks in a single model often requires a particularly diverse feature space. Multimodal (MM) models aim to extract the synergistic predictive potential of multiple data types to create a shared feature space with aligned semantic meaning across inputs of drastically varying sizes (i.e. images, text, sound). Most current MM architectures fuse these representations in parallel, which not only limits their interpretability but also creates a dependency on modality ava

[2] score=0.6011 paper_id=2309.12458 paper-level
title: A Theory of Multimodal Learning
url: https://arxiv.org/abs/2309.12458
categories: cs.LG updated=2023-12-19
preview: Human perception of the empirical world involves recognizing the diverse appearances, or 'modalities', of 

In [ ]:
smoke_questions = [
    'What papers discuss retrieval augmented generation for scientific question answering?',
    'What methods are used for neural information retrieval with dense embeddings?',
    'Which papers use transformers for code generation or program synthesis?',
    'What work studies hallucination or factuality in large language models?',
    'How are graph neural networks used for recommendation or representation learning?',
]

for smoke_question in smoke_questions:
    smoke_results = search_open_arxiv(smoke_question, top_k=5)
    assert len(smoke_results) > 0
    assert all('paper_id' in row and row['text'] and row['arxiv_url'] for row in smoke_results)
    print(f'PASS: {smoke_question} -> {len(smoke_results)} results')


PASS: What papers discuss retrieval augmented generation for scientific question answering? -> 5 results
PASS: What methods are used for neural information retrieval with dense embeddings? -> 5 results
PASS: Which papers use transformers for code generation or program synthesis? -> 5 results
PASS: What work studies hallucination or factuality in large language models? -> 5 results
PASS: How are graph neural networks used for recommendation or representation learning? -> 5 results


In [ ]:
def build_rag_prompt(question: str, sources: list[dict[str, Any]]) -> str:
    context_blocks = []
    for source in sources:
        citation = f"[{source['rank']}] {source['paper_id']} - {source['title']} - {source['arxiv_url']}"
        context_blocks.append(f"{citation}\n{source['text']}")
    context = '\n\n'.join(context_blocks)
    return f"""You answer scientific questions using only the provided source abstracts.
If the sources do not contain enough evidence, say that the evidence is insufficient.
Cite sources with bracketed rank numbers like [1] or [2].
It is acceptable to reason internally, but the final answer must be grounded in the sources.

Question: {question}

Sources:
{context}

Grounded answer:"""

def generate_with_gemini(prompt: str, *, model_name: str = 'gemma-4-31b-it') -> str:
    api_key = os.environ.get('GEMINI_API_KEY')
    if not api_key:
        return 'GEMINI_API_KEY is not set. The retrieval context and prompt are returned for inspection.'

    import urllib.request
    url = f'https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent'
    payload = {
        'contents': [{'role': 'user', 'parts': [{'text': prompt}]}],
        'generationConfig': {'temperature': 0.2, 'maxOutputTokens': 1024},
    }
    request = urllib.request.Request(
        url,
        data=json.dumps(payload).encode('utf-8'),
        headers={'Content-Type': 'application/json', 'x-goog-api-key': api_key},
        method='POST',
    )
    with urllib.request.urlopen(request, timeout=90) as response:
        data = json.loads(response.read().decode('utf-8'))
    parts = data.get('candidates', [{}])[0].get('content', {}).get('parts', [])
    return ''.join(str(part.get('text') or '') for part in parts).strip()

def generate_grounded_answer(question: str, *, top_k: int = 8) -> dict[str, Any]:
    sources = search_open_arxiv(question, top_k=top_k)
    prompt = build_rag_prompt(question, sources)
    answer = generate_with_gemini(prompt, model_name=os.environ.get('GEMINI_MODEL', 'gemma-4-31b-it'))
    return {'answer': answer, 'prompt': prompt, 'sources': sources}

rag_question = 'tensor parallelism for multimodal models??'
rag_result = generate_grounded_answer(rag_question, top_k=8)
print(rag_result['answer'])


*   Question: "tensor parallelism for multimodal models??"
    *   Constraint: Use *only* provided source abstracts.
    *   Constraint: If evidence is insufficient, say so.
    *   Constraint: Cite sources with bracketed rank numbers [1], [2], etc.

    *   [1] MultiModN: Focuses on sequential vs. parallel fusion of latent representations in multimodal networks to handle missing data (MNAR). No mention of "tensor parallelism".
    *   [2] MMPareto: Focuses on gradient conflict between multimodal and unimodal learning objectives. No mention of "tensor parallelism".
    *   [3] A Theory of Multimodal Learning: Theoretical framework on generalization bounds of multimodal vs. unimodal learning. No mention of "tensor parallelism".
    *   [4] Modality Competition: Theoretical explanation of why joint training of multimodal networks can fail (modality competition). No mention of "tensor parallelism".
    *   [5] Improving Multimodal Accuracy: Focuses on modality pre-training and attention. 

## Where embeddings are stored

Embeddings are not saved as raw `.npy` shards. They are stored inside the FAISS index at:

`/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_<run_mode>/faiss/open_arxiv_papers.faiss`

The matching abstract text and metadata are stored in SQLite at:

`/content/drive/MyDrive/scholarrag/open_arxiv_embeddinggemma_fact_check_<run_mode>/metadata/open_arxiv_papers.sqlite`

The manifest at `manifest.json` records the embedding model, prompts, category filters, FAISS index kind, FAISS parameters, row/paper counts, and checkpoint state. Checkpoints live under `checkpoints/`; the notebook keeps only a small number of historical checkpoint copies to avoid filling Drive.
